In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from helpers import *
from implementations import *

In [3]:
x_train = np.load("data/x_train.npy")
y_train = np.load("data/y_train.npy")
x_test = np.load("data/x_test.npy")


In [4]:
print(x_train)

[[5.3000000e+01 1.1000000e+01 1.1162015e+07 ...           nan
            nan 2.0000000e+00]
 [3.3000000e+01 1.2000000e+01 1.2152015e+07 ...           nan
            nan           nan]
 [2.0000000e+01 1.0000000e+01 1.0202015e+07 ... 1.0000000e+00
  2.0000000e+00 2.0000000e+00]
 ...
 [3.9000000e+01 1.0000000e+01 1.0202015e+07 ... 2.0000000e+00
  2.0000000e+00 2.0000000e+00]
 [3.3000000e+01 1.2000000e+01 1.2302015e+07 ...           nan
            nan 2.0000000e+00]
 [3.2000000e+01 9.0000000e+00 9.1220150e+06 ...           nan
            nan 2.0000000e+00]]


In [5]:
vals, counts = np.unique(y_train, return_counts=True)
print(dict(zip(vals, counts)))
y01 = (y_train + 1) / 2 ## Convert -1/1 labels to 0/1 labels


{np.int64(-1): np.int64(299160), np.int64(1): np.int64(28975)}


In [6]:
def preprocess_data(x_tr, x_te, nan_value=0.7):

    ## Remove features with too many NaN values 
    keep = np.isnan(x_tr).mean(axis=0) < nan_value
    x_tr = x_tr[:, keep]
    x_te = x_te[:, keep]

    ## Replace remaining NaN values with median of the feature
    med = np.nanmedian(x_tr, axis=0)
    x_tr = np.where(np.isnan(x_tr), med, x_tr)
    x_te = np.where(np.isnan(x_te), med, x_te)

    ## Delete features with zero variance (division by 0, gives useless values)
    std = x_tr.std(axis=0)
    keep = std > 0
    x_tr = x_tr[:, keep]
    x_te = x_te[:, keep]
    std = std[keep]

    ## Standardize features
    mean = x_tr.mean(axis=0)
    x_tr = (x_tr - mean) / std
    x_te = (x_te - mean) / std

    ## Add bias column (column of ones) to keep w0, the model is not forced to go through the origin
    x_tr = np.c_[np.ones(len(x_tr)), x_tr]
    x_te = np.c_[np.ones(len(x_te)), x_te]

    return x_tr, x_te







In [7]:
## Split the training data into a training and validation set (x_train, x_val, y_train, y_val)
def split_data(x,y, ratio=0.8, seed=1):
    np.random.seed(seed)
    idx = np.random.permutation(len(y))
    cut = int(len(y) * ratio)
    return x[idx[:cut]], x[idx[cut:]], y[idx[:cut]], y[idx[cut:]]


x_tr_split, x_te_split, y_tr_split, y_te_split = split_data(x_train, y01)


In [8]:
## Process the data (remove NaN, standardize, add bias column)
x_tr_p, x_te_p = preprocess_data(x_tr_split, x_te_split)


In [9]:
## Predict the labels of the test set using logistic regression
def predict_labels_logistic(x, w, threshold=0.5):
    return (sigmoid_function(x @ w) >= threshold).astype(int)

def predict_labels(x, w, threshold=0.22):
    return (x @ w >= threshold).astype(int)

## Given the true labels and predicted labels, compute the F1 score
def f1_score(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
        
    return 2 * tp / (2*tp + fn + fp) if tp > 0 else 0.0


w_least_squares, loss = least_squares(y_tr_split, x_tr_p)
predictions = predict_labels(x_te_p, w_least_squares)
f1 = f1_score(y_te_split, predictions)
print(f"F1 score for least squares: {f1:.4f}")


F1 score for least squares: 0.4054


In [10]:
def build_k_indices(y, k_fold, seed):
    num_row = y.shape[0]
    interval = int(num_row / k_fold)
    np.random.seed(seed)
    indices = np.random.permutation(num_row)
    k_indices = [indices[k * interval : (k + 1) * interval] for k in range(k_fold)]
    return np.array(k_indices)

In [11]:
def build_poly(x, degree):
    poly = np.ones((len(x), 1))
    for deg in range(1, degree + 1):
        poly = np.c_[poly, np.power(x, deg)]
    return poly

In [12]:
def cross_validation(y, tx, k_indices, k, lambda_, degree, thresholds):
    te_indice = k_indices[k]
    tr_indice = k_indices[~(np.arange(k_indices.shape[0]) == k)].reshape(-1)
    
    # ridge regression
    w, _ = ridge_regression(y[tr_indice], tx[tr_indice], lambda_)
    scores = tx[te_indice] @ w
    return np.array([f1_score(y[te_indice], (scores >= t).astype(int)) for t in thresholds])


In [13]:
def best_degree_selection(y, x, degrees, k_fold, lambdas, seed=1,
                          thresholds=np.linspace(0.05, 0.95, 19)):
    # split data in k fold
    k_indices = build_k_indices(y, k_fold, seed)

    best_lambdas, best_thresholds, best_f1s = [], [], []

    for degree in degrees:
        tx = build_poly(x, degree)
        f1_per_lambda, thr_per_lambda = [], []

        for lambda_ in lambdas:
            # tableau (k_fold, nb_seuils)
            f1s = np.array([cross_validation(y, tx, k_indices, k, lambda_, degree, thresholds)
                            for k in range(k_fold)])
            mean_f1 = f1s.mean(axis=0)          # F1 moyen pour chaque seuil
            i = np.argmax(mean_f1)
            f1_per_lambda.append(mean_f1[i])
            thr_per_lambda.append(thresholds[i])

        ind_lambda_opt = np.argmax(f1_per_lambda)
        best_lambdas.append(lambdas[ind_lambda_opt])
        best_thresholds.append(thr_per_lambda[ind_lambda_opt])
        best_f1s.append(f1_per_lambda[ind_lambda_opt])
        print(f"degree={degree}: lambda={lambdas[ind_lambda_opt]:.1e}, "
              f"seuil={thr_per_lambda[ind_lambda_opt]:.2f}, F1={f1_per_lambda[ind_lambda_opt]:.4f}")

    ind_best_degree = np.argmax(best_f1s)
    return (int(degrees[ind_best_degree]),
            float(best_lambdas[ind_best_degree]),
            float(best_thresholds[ind_best_degree]),
            float(best_f1s[ind_best_degree]))


In [14]:
best_deg, best_lambda, best_threshold, best_f1 = best_degree_selection(y_tr_split, x_tr_p, degrees=[1, 2, 3], lambdas=np.logspace(-8, -1, 8), k_fold=5)

tx_tr = build_poly(x_tr_p, best_deg)
tx_te = build_poly(x_te_p, best_deg)

w, _ = ridge_regression(y_tr_split, tx_tr, best_lambda)

pred = (tx_te @ w >= best_threshold).astype(int)
print(f"F1 CV = {best_f1:.4f} | F1 test = {f1_score(y_te_split, pred):.4f}")

degree=1: lambda=1.0e-07, seuil=0.20, F1=0.4080
degree=2: lambda=1.0e-04, seuil=0.20, F1=0.4173
degree=3: lambda=1.0e-08, seuil=0.20, F1=0.4179
F1 CV = 0.4179 | F1 test = 0.4121


In [16]:
from model_selection import f1_score
scores = tx_tr @ w   # w = poids du modèle final
ts = np.quantile(scores, np.linspace(0.3, 0.99, 100))
f1s = [f1_score(y01, (scores >= t).astype(int)) for t in ts]
plt.figure(figsize=(4, 2.8))
plt.plot(np.mean(scores[:, None] >= ts, axis=0), f1s)
plt.xlabel("Fraction predicted positive"); plt.ylabel("F1")
plt.tight_layout(); plt.savefig("threshold.pdf")

ValueError: operands could not be broadcast together with shapes (328135,) (262508,) 